In [36]:
!pip install lxml


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Fase 1: Descarga y almacenamiento RAW

In [37]:
import pandas as pd
import requests

from io import StringIO
from time import sleep


def download_bref_table(table_type,
                        start_year=2001,
                        end_year=2025,
                        pause=5):

    dfs = []

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/137.0 Safari/537.36"
        )
    }

    for season in range(start_year, end_year + 1):

        url = (
            f"https://www.basketball-reference.com/"
            f"leagues/NBA_{season}_{table_type}.html"
        )

        try:

            response = requests.get(
                url,
                headers=headers,
                timeout=30
            )

            if response.status_code == 429:

                print(
                    f"ERROR {season}: "
                    "HTTP 429 (Too Many Requests)"
                )

                continue

            response.raise_for_status()

            df = pd.read_html(
                StringIO(response.text)
            )[0]

            # eliminar encabezados repetidos
            if isinstance(df.columns, pd.MultiIndex):

                first_col = df.columns[0]

                df = df[df[first_col] != "Rk"]

            else:

                if "Rk" in df.columns:
                    df = df[df["Rk"] != "Rk"]

            df["Season"] = (
                f"{season-1}-{str(season)[-2:]}"
            )

            dfs.append(df)

            print(f"OK {season}")

            sleep(pause)

        except Exception as e:

            print(
                f"ERROR {season}: {e}"
            )

    if len(dfs) == 0:

        raise ValueError(
            "No se descargó ninguna temporada."
        )

    return pd.concat(
        dfs,
        ignore_index=True
    )

### PER GAME

In [ ]:
per_game = download_bref_table("per_game")

per_game.to_csv(
    "nba_per_game.csv",
    index=False,
    encoding="utf-8-sig"
)

OK 2001
OK 2002
OK 2003
OK 2004
OK 2005
OK 2006
OK 2007
OK 2008
OK 2009
OK 2010
OK 2011
OK 2012
OK 2013
OK 2014
OK 2015
OK 2016
OK 2017
OK 2018
OK 2019
OK 2020
OK 2021
OK 2022
OK 2023
OK 2024
OK 2025


### ADVANCED

In [ ]:
advanced = download_bref_table("advanced")

advanced.to_csv(
    "nba_advanced.csv",
    index=False,
    encoding="utf-8-sig"
)

OK 2001
OK 2002
OK 2003
OK 2004
OK 2005
OK 2006
OK 2007
OK 2008
OK 2009
OK 2010
OK 2011
OK 2012
OK 2013
OK 2014
OK 2015
OK 2016
OK 2017
OK 2018
OK 2019
OK 2020
OK 2021
ERROR 2022: ("Connection broken: ConnectionResetError(10054, 'Se ha forzado la interrupción de una conexión existente por el host remoto', None, 10054, None)", ConnectionResetError(10054, 'Se ha forzado la interrupción de una conexión existente por el host remoto', None, 10054, None))
OK 2023
OK 2024
OK 2025


Solucionamos el problema con la descarga del 2022 y lo hacemos a parte sólo de ese año.

In [75]:
# Cargamos el CSV que ya había guardado
advanced = pd.read_csv(
    "nba_advanced.csv"
)

In [76]:
# Descarga única del 2022

advanced_2022 = download_bref_table(
    "advanced",
    start_year=2022,
    end_year=2022,
    pause=10
)

OK 2022


In [78]:
# Añadimosla temporada faltante
advanced = pd.concat(
    [advanced, advanced_2022],
    ignore_index=True
)

In [80]:
# Guardamos de nuevo
advanced.to_csv(
    "nba_advanced.csv",
    index=False,
    encoding="utf-8-sig"
)

In [81]:
# Eliminamos posibles duplicados

advanced = advanced.drop_duplicates()

In [82]:
# Verificamos que ya están las 25 temporadas

sorted(
    advanced["Season"].unique()
)

['2000-01',
 '2001-02',
 '2002-03',
 '2003-04',
 '2004-05',
 '2005-06',
 '2006-07',
 '2007-08',
 '2008-09',
 '2009-10',
 '2010-11',
 '2011-12',
 '2012-13',
 '2013-14',
 '2014-15',
 '2015-16',
 '2016-17',
 '2017-18',
 '2018-19',
 '2019-20',
 '2020-21',
 '2021-22',
 '2022-23',
 '2023-24',
 '2024-25']

In [83]:
test = pd.read_csv(
    "nba_advanced.csv"
)

print(
    test["Season"].nunique()
)

print(
    "2021-22" in test["Season"].astype(str).unique()
)

25
True


### SHOOTING

In [ ]:
shooting = download_bref_table("shooting")

shooting.to_csv(
    "nba_shooting.csv",
    index=False,
    encoding="utf-8-sig"
)

OK 2001
OK 2002
OK 2003
OK 2004
OK 2005
OK 2006
OK 2007
OK 2008
OK 2009
OK 2010
OK 2011
OK 2012
OK 2013
OK 2014
OK 2015
OK 2016
OK 2017
OK 2018
OK 2019
OK 2020
OK 2021
OK 2022
OK 2023
OK 2024
OK 2025


### TOTALS

In [41]:
totals = download_bref_table("totals")

totals.to_csv(
    "nba_totals.csv",
    index=False,
    encoding="utf-8-sig"
)

OK 2001
OK 2002
OK 2003
OK 2004
OK 2005
OK 2006
OK 2007
OK 2008
OK 2009
OK 2010
OK 2011
OK 2012
OK 2013
OK 2014
OK 2015
OK 2016
OK 2017
OK 2018
OK 2019
OK 2020
OK 2021
OK 2022
OK 2023
OK 2024
OK 2025


## ** CORRECCIÓN POR PROBLEMAS CON EL CODING DE LOS NOMBRES ** SE AÑADE POSTERIORMENTE

In [46]:
import pandas as pd

In [53]:
per_game = pd.read_csv(r"Datasets\Jugador_temporada - Basketball Reference\nba_per_game.csv")
totals = pd.read_csv(r"Datasets\Jugador_temporada - Basketball Reference\nba_totals.csv")
advanced = pd.read_csv(r"Datasets\Jugador_temporada - Basketball Reference\nba_advanced.csv")
shooting = pd.read_csv(r"Datasets\Jugador_temporada - Basketball Reference\nba_shooting.csv", header=[0, 1])

In [54]:
def normalize_player_names(text):

    if pd.isna(text):
        return text

    try:
        return text.encode("latin1").decode("utf-8")
    except:
        return text


for df in [per_game, totals, advanced]:

    df["Player"] = df["Player"].apply(normalize_player_names)

## Fase 2: Limpieza individual de cada tabla

#### Limpieza básica: Per Game, Advanced y Totals.

In [55]:
def clean_player_table(df):

    df = df.copy()

    # quitar asteriscos HOF/All-Star
    df["Player"] = (
        df["Player"]
        .str.replace("*", "", regex=False)
        .str.strip()
    )

    # nombres de equipo consistentes
    df["Team"] = (
        df["Team"]
        .astype(str)
        .str.strip()
    )

    return df

In [56]:
per_game = clean_player_table(per_game)
advanced = clean_player_table(advanced)
totals = clean_player_table(totals)

#### Limpieza Shooting

In [57]:
# Realizamos una copia por si fuera necesario volver a atrás
shooting_raw = shooting.copy()

In [58]:
# Aplanamos las columnas
shooting.columns = [
    f"{c1}_{c2}"
    for c1, c2 in shooting.columns
]

In [59]:
shooting = shooting.rename(columns={
    "Unnamed: 1_level_0_Player": "Player",
    "Unnamed: 2_level_0_Age": "Age",
    "Unnamed: 3_level_0_Team": "Team",
    "Unnamed: 4_level_0_Pos": "Pos",
    "Unnamed: 5_level_0_G": "G",
    "Unnamed: 6_level_0_GS": "GS",
    "Unnamed: 7_level_0_MP": "MP",
    "Unnamed: 8_level_0_FG%": "FGPct",
    "Unnamed: 9_level_0_Dist.": "AvgShotDistance",
    "Season_Unnamed: 31_level_1": "Season"
})

In [60]:
print(shooting.shape)

print(shooting["Season"].nunique())

print(shooting["Player"].nunique())

print(
    shooting[["Player","Season"]].head()
)

(15422, 32)
25
2474
           Player   Season
0  Michael Finley  2000-01
1  Antoine Walker  2000-01
2  Antawn Jamison  2000-01
3   Anthony Mason  2000-01
4     Gary Payton  2000-01


In [61]:
shooting["Player"] = (
    shooting["Player"]
    .str.replace("*", "", regex=False)
    .str.strip()
)

#### Comprobación

In [62]:
print(per_game.shape)
print(advanced.shape)
print(totals.shape)
print(shooting.shape)

(15422, 32)
(15422, 30)
(15422, 33)
(15422, 32)


In [63]:
for name, df in {
    "per_game": per_game,
    "advanced": advanced,
    "totals": totals,
    "shooting": shooting
}.items():

    print("\n", name)

    print(
        df[["Player","Season"]]
        .duplicated()
        .sum()
    )


 per_game
3173

 advanced
3173

 totals
3173

 shooting
3173


## ** CORRECCIÓN POR PROBLEMAS CON EL CODING DE LOS NOMBRES ** SE AÑADE POSTERIORMENTE

In [64]:
shooting["Player"] = (
    shooting["Player"]
    .apply(normalize_player_names)
)

In [65]:
print(
    shooting[
        shooting["Player"].str.contains("Peja", na=False)
    ][["Player"]].head()
)

print(
    shooting[
        shooting["Player"].str.contains("Hedo", na=False)
    ][["Player"]].head()
)

               Player
33    Peja Stojaković
586   Peja Stojaković
1116  Peja Stojaković
1524  Peja Stojaković
2179  Peja Stojaković
             Player
244   Hedo Türkoğlu
667   Hedo Türkoğlu
1291  Hedo Türkoğlu
1669  Hedo Türkoğlu
2286  Hedo Türkoğlu


### Exportar los csv limpios

In [67]:
per_game.to_csv(
    "nba_per_game_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

totals.to_csv(
    "nba_totals_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

advanced.to_csv(
    "nba_advanced_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

shooting.to_csv(
    "nba_shooting_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

## Fase 3: Resolver el problema TOT (jugadores traspasados a media temporada)

In [68]:
def keep_multiteam_or_single(df):

    result = []

    for _, group in df.groupby(
        ["Player", "Season"],
        sort=False
    ):

        multi = group[
            group["Team"]
            .astype(str)
            .str.match(r"^\dTM$")
        ]

        if len(multi) > 0:
            result.append(multi)

        else:
            result.append(group)

    return pd.concat(
        result,
        ignore_index=True
    )

In [69]:
per_game_clean = keep_multiteam_or_single(per_game)
advanced_clean = keep_multiteam_or_single(advanced)
totals_clean = keep_multiteam_or_single(totals)
shooting_clean = keep_multiteam_or_single(shooting)

AUDITORÍA

In [70]:
for name, df in {
    "per_game_clean": per_game_clean,
    "advanced_clean": advanced_clean,
    "totals_clean": totals_clean,
    "shooting_clean": shooting_clean
}.items():

    print(name)

    print(
        df[
            ["Player","Age","Season"]
        ]
        .duplicated()
        .sum()
    )

per_game_clean
0
advanced_clean
0
totals_clean
0
shooting_clean
0


## Fase 4: Crear llaves

In [71]:
keys = ["Player", "Age", "Season"]

for name, df in {
    "per_game": per_game_clean,
    "advanced": advanced_clean,
    "totals": totals_clean,
    "shooting": shooting_clean
}.items():

    print(
        name,
        df[keys]
        .drop_duplicates()
        .shape[0]
    )

per_game 12252
advanced 12252
totals 12252
shooting 12252


## Fase 5: Merge maestro

In [72]:
print(len(per_game_clean))
print(len(advanced_clean))
print(len(totals_clean))
print(len(shooting_clean))

12252
12252
12252
12252


### MERGE 1

In [73]:
master = per_game_clean.merge(
    advanced_clean,
    on=["Player", "Age", "Season"],
    how="inner",
    suffixes=("", "_adv")
)

print(master.shape)

(12252, 59)


### MERGE 2

In [74]:
master = master.merge(
    totals_clean,
    on=["Player", "Age", "Season"],
    how="inner",
    suffixes=("", "_tot")
)

print(master.shape)

(12252, 89)


### MERGE 3

In [75]:
master = master.merge(
    shooting_clean,
    on=["Player", "Age", "Season"],
    how="inner",
    suffixes=("", "_shoot")
)

print(master.shape)

(12252, 118)


### COMPROBACIÓN

In [76]:
print(master.shape)

print(
    master[
        ["Player", "Age", "Season"]
    ].duplicated().sum()
)

# Debe dar 0

(12252, 118)
0


### Guardamos el dataset maestro

In [77]:
master.to_csv(
    "nba_master_dataset_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

In [78]:
master.to_parquet(
    "nba_master_dataset_v1.parquet",
    index=False
)